In [1]:
import base64
import os

from openai import OpenAI

# DeepSeek-V4.1-Flash 的官方调用名是 deepseek-flash
# deepseek-v4.1-flash 不是 API 模型名，官方不会识别
MODEL = "deepseek-flash"

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

# 文字输出
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "用中文解释AI大模型是如何工作的"}],
)

print(response.choices[0].message.content)

可以用一句话概括：

**AI 大模型本质上是一个超大的神经网络，通常基于 Transformer 架构。它通过阅读海量文本，学会“根据前文预测下一个词元”，然后不断把预测出来的词元接回前文，循环生成整段回答。**

可以把它想成一个“超级文字接龙高手”。

---

## 1. 核心任务：预测下一个词元

大模型处理文本时，不是直接按“字”或“词”处理，而是先切成 **token（词元）**。一个 token 可能是一个字、一个词的一部分，或一个常见词组。

例如：

> 输入：中国的首都是  
> 模型预测下一个 token 最可能是：北京

模型内部会给词表中每个 token 算一个概率，比如：

- 北京：92%
- 上海：3%
- 南京：1%
- 其他：4%

然后根据这些概率选一个 token。选出来后，把它接到输入后面：

> 中国的首都是北京

再预测下一个 token，可能是“，”“它”“是”等。如此循环，直到生成完整回答。

所以大模型生成文本的过程是 **自回归生成**：一次一个 token，不断往后接。

---

## 2. 内部结构：Transformer 神经网络

大模型的核心通常是 **Transformer**。它由很多层神经网络堆叠而成，每层主要包括：

- **自注意力机制**：让每个词去看句子里其他词，判断应该关注谁。  
  比如“小明把书放进书包，因为**它**很重”，模型通过注意力机制判断“它”可能指书包或书。
- **前馈网络**：对信息做非线性变换，提取更复杂的模式。
- **残差连接和归一化**：帮助深层网络稳定训练。
- **位置编码**：让模型知道词的顺序。

经过几十层甚至上百层处理后，模型把输入的文本变成一系列高维向量。最后通过输出层，计算出下一个 token 的概率分布。

模型里的 **参数** 就是这些神经网络中的权重。大模型有几十亿到上万亿个参数。你可以把参数想成无数个旋钮，训练就是不断调整这些旋钮，让预测更准确。

---

## 3. 训练过程：从“会续写”到“会对话”

大模型通常分几个阶段训练。

### 第一阶段：预训练

给模型喂海量文本：网页、书籍、论文、代码、百科等。  
训练目标是：**给定前面的文本，预测下一个 token**。

比如：

> “今天天气真___”

模型应该

In [2]:
# 图像理解：本地图片转成 data URL 后和文字一起放入 content 列表
with open("dog_and_girl.jpeg", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                },
                {"type": "text", "text": "帮我解释下这张照片"},
            ],
        }
    ],
)

print(response.choices[0].message.content)

这张照片是一张非常温馨、充满治愈感的人宠互动摄影作品。以下是详细的画面解析：

**1. 画面主体与动作**
*   **人物**：一位年轻女性坐在沙滩上，穿着深色格子衬衫和牛仔裤，长发披肩。她面带温柔的笑容，正注视着面前的狗狗。
*   **狗狗**：一只浅黄色的拉布拉多犬（或类似犬种），佩戴着带有彩色花纹的胸背带，旁边沙滩上还放着一根红色牵引绳。
*   **互动**：狗狗坐直身体，抬起左前爪与女生的手掌相触（像是在击掌或握手）。这个动作展现了极高的默契和亲昵感。

**2. 环境与背景**
*   背景是一片开阔的海滩，远处是轻柔的海浪和广阔的海平线。
*   沙滩上有着自然的起伏和脚印，质感细腻。

**3. 光线与色彩（摄影美学）**
*   照片采用**逆光（黄金时刻）**拍摄，光源（可能是日出或日落）在女生身后，给她的头发镀上了一层金黄色的光晕。
*   整体色调是非常温暖的暖橘色与柔和的米白色。高调的曝光让画面显得轻盈、通透。

**4. 情感与意境**
*   这张照片传达的主题是**陪伴、信任与纯粹的爱**。
*   画面语言非常平和，一人一狗在海边伴着夕阳（或晨光）互动，将人与宠物之间那种跨越物种的情感纽带具象化了。没有喧闹，只有安静的陪伴，给人非常放松、温暖、治愈的感觉。

总而言之，这是一张极具生活气息且构图讲究的摄影作品，完美捕捉了人与宠物共度美好时光的瞬间。


In [3]:
# 视频理解
# DeepSeek 只支持图片多模态，不能像 Gemini 那样直接上传 mp4。
# 这里从视频里均匀抽帧，再按图片理解的方式送给模型。
import cv2

video_path = "car.mp4"
frame_num = 6

cap = cv2.VideoCapture(video_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if total <= 0:
    raise ValueError("无法读取视频帧")

indexes = [int(i * (total - 1) / (frame_num - 1)) for i in range(frame_num)]
content = [{"type": "text", "text": "详细描述视频里发生了什么？如果有对话，请把关键对话提取出来。"}]

print("正在从视频抽帧...")
for idx in indexes:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        raise ValueError(f"读取第 {idx} 帧失败")
    ok, buf = cv2.imencode(".jpg", frame)
    if not ok:
        raise ValueError(f"编码第 {idx} 帧失败")
    frame_b64 = base64.b64encode(buf.tobytes()).decode("utf-8")
    content.append(
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{frame_b64}"},
        }
    )
cap.release()
print(f"抽帧完成，共 {frame_num} 帧，开始推理...")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": content}],
)

print(response.choices[0].message.content)

正在从视频抽帧...
抽帧完成，共 6 帧，开始推理...
根据视频截图，整个视频主要记录了车主（视频中的男士）在展示和评估车辆剐蹭受损情况，并讨论维修方案的过程。详细描述如下：

**视频内容详细描述：**

1. **车辆受损特写（图1-图2）**：视频开头展示了车辆右后侧轮胎和轮眉处的受损情况。黑色轮眉和轮胎侧壁上有明显的白色摩擦痕迹，受损较为严重。
2. **现场评估与定损（图3-图5）**：镜头切换到车主（戴眼镜的男士）站在车辆旁边。他不断弯腰或蹲下查看车漆损伤，用手比划受损范围，并向身旁的一位女士（或拍摄者）说明情况。他判断损伤不仅限于轮眉，还波及到了车身侧面的车漆（一直延伸到门把手附近），因此判断需要进行**钣金修复**。同时，他告知对方“咱们没有车损险”，意味着需要自费修理。
3. **事故现场回放（图6）**：画面转到地下车库。车主站在一面有明显掉漆、破损的承重墙或柱子旁，指着墙面。这表明这里是事故的发生地点，他正在解释或展示车辆是与这根柱子发生碰撞才导致了上述的车辆损伤（他特意指出墙面上的损坏痕迹不仅是他这次撞的，暗示这块区域比较狭窄或本身就有受损）。

**关键对话提取（根据字幕）：**

*   “直接撞墙上了”（描述事故原因及过程）
*   “这肯定得钣金”（对车辆受损程度的维修结论）
*   “到门把手这块”（描述车漆刮蹭的扩散范围）
*   “咱们没有车损险”（提示维修费用的承担方式）
*   “这不仅是我刚才磕的这个地方”（解释地下车库现场的墙面受损情况，说明不是单次撞击造成的全部损伤）

**总结：** 这是一个典型的“用车事故记录”或“汽车维修经验分享”的短视频片段，主要讲述了车主在没有车损险的情况下撞墙，并查看车辆钣金受损情况的过程。
